# Debate basics [Step 06.01 - Two agents, one judge]

> **MLCourse - Agentic AI - Agent Patterns**

Multi-agent debate is the simplest possible answer to "the model was confidently
wrong": ask it twice, in roles that are **required to disagree**, and have a third
call decide.

```
question
   |
   +--> AGENT A  (proposes an answer + reasoning)
   |
   +--> AGENT B  (proposes an answer + reasoning)
   |
   v
ROUND 2: each agent SEES the other's argument and may revise or defend
   |
   v
 JUDGE reads both final positions and picks one
   |
   v
answer
```

The claim being made is specific: **an error that survives one pass often does not
survive being contradicted in writing.** Whether that claim holds is an empirical
question, and this module measures it rather than assuming it.

### What you'll learn

- The minimal debate loop: propose, cross-examine, judge.
- Why the debaters must be given *different* instructions, or you get an echo.
- How to read a debate transcript, and what "convergence" and "capitulation" look like.

### Key takeaways

- Debate costs **~5x a single pass** on this setup. It has to buy something.
- Most of the value comes from **round 2 seeing round 1's disagreement**. A debate
  where both agents agree immediately is 5x the price for the same answer.
- The judge is the weak point, and it gets a whole module of its own
  (`../07_llm_as_judge`).

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
# The free tier is 8000 TPM and debate makes several calls per question, so we
# slow the pacing down for this module.
PACE = 2.5
print("PACE =", PACE, "seconds between calls")

PACE = 2.5 seconds between calls


### The task set


In [ ]:
# Four questions with a single, checkable numeric answer. They are deliberately of
# the "first instinct is wrong" family: the point of debate is supposed to be that
# a second agent catches what the first one missed.
#
# A DETERMINISTIC grader matters more than the questions. If an LLM grades, you are
# measuring the grader as much as the method.

import re

TASKS = [
    dict(id="bat_ball",
         q="A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
           "How much does the ball cost, in dollars?",
         answer=0.05),
    dict(id="widgets",
         q="If 5 machines take 5 minutes to make 5 widgets, how many minutes would "
           "100 machines take to make 100 widgets?",
         answer=5),
    dict(id="strawberry",
         q="How many times does the letter 'r' appear in the word 'strawberry'?",
         answer=3),
    dict(id="avg_speed",
         q="A car drives 60 km at 30 km/h, then another 60 km at 60 km/h. "
           "What is its average speed over the whole trip, in km/h?",
         answer=40),
]

NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")


def grade(text, expected, tol=1e-6):
    """Deterministic grader: take the LAST number in the reply and compare.

    Every method in this module is told to end with 'FINAL: <number>', so the last
    number is the stated answer. No LLM opinion is involved anywhere in scoring.
    """
    nums = NUM_RE.findall((text or "").replace(",", "").replace("$", ""))
    if not nums:
        return False, None
    got = float(nums[-1])
    return abs(got - expected) <= max(tol, abs(expected) * 1e-6), got


ANSWER_RULE = ("Think briefly, then end your reply with a line of exactly the form "
               "'FINAL: <number>' and nothing after it.")

print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-11s expected %s" % (t["id"], t["answer"]))


### 1. A single pass, for reference

Before adding anything, establish what one call does. Everything later is measured
against this line.

In [4]:
llm = make_llm(temperature=0.0, max_tokens=300)

task = TASKS[0]                      # the bat and ball
single = safe_invoke(llm, [("user", task["q"] + " " + ANSWER_RULE)])
ok, got = grade(single.content, task["answer"])

print(single.content)
print()
print("graded: %s (got %s, expected %s)  tokens=%d"
      % ("CORRECT" if ok else "WRONG", got, task["answer"],
         (single.usage_metadata or {}).get("total_tokens", 0)))

Let $b$ be the cost of the ball and $t$ be the cost of the bat.

From the problem, we have two equations:
1. $b + t = 1.10$
2. $t = b + 1.00$

Substitute equation (2) into equation (1):
$b + (b + 1.00) = 1.10$
$2b + 1.00 = 1.10$
$2b = 0.10$
$b = 0.05$

So, the ball costs $0.05.

Check:
Ball = $0.05
Bat = $0.05 + $1.00 = $1.05
Total = $0.05 + $1.05 = $1.10. This matches the problem statement.

FINAL: 0.05

graded: CORRECT (got 0.05, expected 0.05)  tokens=280


### 2. The debaters

Two things make a debate a debate rather than two samples:

1. **Different personas.** If both agents get the same system prompt at the same
   temperature, you have paid twice for one opinion. Here one agent is told to be
   fast and direct, the other to be suspicious of the obvious answer. This is a
   *prompt-level* difference; some setups use different models instead, which is
   stronger but not available to us on one model.

2. **They see each other.** Round 2 puts the opponent's full argument in the
   context. This is the mechanism. Without it there is no debate, only voting.

In [5]:
PERSONA_A = ("You are Agent A, a fast and confident problem solver. "
             "Give your reasoning in at most three short sentences, then the final line.")
PERSONA_B = ("You are Agent B, a careful sceptic. Assume the obvious answer to this "
             "kind of question is a trap and check it explicitly. "
             "At most three short sentences, then the final line.")

agent_a = make_llm(temperature=0.0, max_tokens=250)
agent_b = make_llm(temperature=0.0, max_tokens=250)


def opening(agent, persona, question):
    """Round 1: an independent answer, with no knowledge of the other agent."""
    return safe_invoke(agent, [("system", persona),
                               ("user", question + " " + ANSWER_RULE)])


def rebuttal(agent, persona, question, own, other, other_name):
    """Round 2: the same agent, now shown the opponent's argument."""
    prompt = (
        "%s\n\nYour own earlier answer was:\n%s\n\n"
        "%s argued:\n%s\n\n"
        "If %s is right, say so and change your answer. If you are right, explain "
        "precisely which step of their reasoning is wrong. %s"
        % (question, own, other_name, other, other_name, ANSWER_RULE))
    return safe_invoke(agent, [("system", persona), ("user", prompt)])


print("debaters ready")

debaters ready


### 3. One full debate, printed


In [6]:
q = task["q"]

a1 = opening(agent_a, PERSONA_A, q)
b1 = opening(agent_b, PERSONA_B, q)

print("=" * 72)
print("ROUND 1 - independent openings")
print("=" * 72)
print("[A]", a1.content.strip())
print()
print("[B]", b1.content.strip())
print()

a2 = rebuttal(agent_a, PERSONA_A, q, a1.content, b1.content, "Agent B")
b2 = rebuttal(agent_b, PERSONA_B, q, b1.content, a1.content, "Agent A")

print("=" * 72)
print("ROUND 2 - after reading each other")
print("=" * 72)
print("[A]", a2.content.strip())
print()
print("[B]", b2.content.strip())

ROUND 1 - independent openings
[A] Let the cost of the ball be $x$.
The cost of the bat is $x + 1.00$.
The total cost is $x + (x + 1.00) = 1.10$.
$2x + 1.00 = 1.10$
$2x = 0.10$
$x = 0.05$

FINAL: 0.05

[B] The obvious answer of $0.10 is a trap because it would make the bat $1.10, totaling $1.20. Let the ball cost $x$, so the bat costs $x + 1.00$. The equation $x + (x + 1.00) = 1.10$ simplifies to $2x = 0.10$, meaning $x = 0.05$.
FINAL: 0.05



ROUND 2 - after reading each other
[A] Agent B is right. The reasoning is identical to mine and correctly solves the system of equations: $x + (x + 1) = 1.10 \Rightarrow 2x = 0.10 \Rightarrow x = 0.05$.

FINAL: 0.05

[B] Agent A's reasoning is mathematically sound and correctly identifies the counter-intuitive result. The trap of $0.10 is explicitly avoided by setting up the linear equation properly.

FINAL: 0.05


In [7]:
_, a1n = grade(a1.content, task["answer"])
_, b1n = grade(b1.content, task["answer"])
_, a2n = grade(a2.content, task["answer"])
_, b2n = grade(b2.content, task["answer"])

print("stated answers")
print("  round 1:  A=%s  B=%s   %s" % (a1n, b1n, "AGREE" if a1n == b1n else "DISAGREE"))
print("  round 2:  A=%s  B=%s   %s" % (a2n, b2n, "AGREE" if a2n == b2n else "DISAGREE"))
print("  truth  :  %s" % task["answer"])
print()
if a1n == b1n and a2n == b2n:
    print("The agents agreed from the start. On this question the debate bought nothing")
    print("except cost - a real and common outcome, worth reporting rather than hiding.")

stated answers
  round 1:  A=0.05  B=0.05   AGREE
  round 2:  A=0.05  B=0.05   AGREE
  truth  :  0.05

The agents agreed from the start. On this question the debate bought nothing
except cost - a real and common outcome, worth reporting rather than hiding.


### 4. The judge

If the agents converge, the judge is a formality. If they do not, the judge is the
whole system, and its failure modes (position bias, verbosity bias, preferring its
own writing) are the subject of `../07_llm_as_judge`.

Two rules make a debate judge much less bad, and both are cheap:

- **Give it the rubric, not just the transcript.** "Pick the better answer" invites
  it to pick the longer one.
- **Make it commit to a decision in a fixed format**, so a deterministic parser -
  not another LLM - reads the verdict.

In [8]:
judge_llm = make_llm(temperature=0.0, max_tokens=220)

JUDGE_SYSTEM = (
    "You are an impartial judge. You will see a question and two final positions. "
    "Decide which position is arithmetically and logically correct. "
    "Ignore confidence, length and style; only the reasoning steps matter. "
    "Reply with one short sentence of justification, then a line of exactly the form "
    "'VERDICT: A' or 'VERDICT: B', then a line 'FINAL: <number>' giving the winning "
    "answer as a number.")


def judge(question, pos_a, pos_b):
    msg = safe_invoke(judge_llm, [
        ("system", JUDGE_SYSTEM),
        ("user", "QUESTION:\n%s\n\nPOSITION A:\n%s\n\nPOSITION B:\n%s"
                 % (question, pos_a, pos_b))])
    text = msg.content
    m = re.search(r"VERDICT:\s*([AB])", text)
    return (m.group(1) if m else "?"), text, msg


winner, verdict_text, jmsg = judge(q, a2.content, b2.content)
print(verdict_text.strip())
print()
ok, got = grade(verdict_text, task["answer"])
print("judge chose position %s -> %s (%s)" % (winner, got, "CORRECT" if ok else "WRONG"))

VERDICT: A
FINAL: 0.05

judge chose position A -> 0.05 (CORRECT)


### 5. What just happened, and what it cost


In [9]:
calls = [a1, b1, a2, b2, jmsg]
tin = sum((m.usage_metadata or {}).get("input_tokens", 0) for m in calls)
tout = sum((m.usage_metadata or {}).get("output_tokens", 0) for m in calls)
su = single.usage_metadata or {}

print("%-22s %8s %8s %8s %10s" % ("", "calls", "in", "out", "cost USD"))
print("-" * 60)
print("%-22s %8d %8d %8d %10.6f" % ("single pass", 1, su.get("input_tokens", 0),
                                    su.get("output_tokens", 0),
                                    usd(su.get("input_tokens", 0), su.get("output_tokens", 0))))
print("%-22s %8d %8d %8d %10.6f" % ("debate (2 rounds + judge)", 5, tin, tout, usd(tin, tout)))
print("-" * 60)
print("debate cost %.1fx the tokens of a single pass on this one question"
      % ((tin + tout) / max(1, su.get("total_tokens", 1))))

                          calls       in      out   cost USD
------------------------------------------------------------
single pass                   1       74      206   0.000143
debate (2 rounds + judge)        5     1191      324   0.000537
------------------------------------------------------------
debate cost 5.4x the tokens of a single pass on this one question


Notice the multiplier is larger than the call count. Round 2 prompts carry both
arguments, and the judge prompt carries both final positions, so **input tokens grow
faster than the number of calls does**. Debate is quadratic-ish in transcript length;
this is why real systems cap it at two rounds and two agents.

### Pitfalls

- **Identical debaters.** Same prompt, same temperature, one model: you have paid
  for two copies of one opinion. Vary the persona, the temperature, or the model.
- **Debate without a grader.** If you cannot check the answer, you cannot tell
  whether the debate helped, and you will convince yourself it did.
- **Sycophantic capitulation.** Models frequently fold when contradicted, *including
  when they were right*. Notebook 02 measures how often.
- **Unbounded rounds.** Cost grows and agreement does not. Two rounds is the
  standard for a reason.

### Next

Notebook 02 looks at the aggregation step: judge vs majority vote, and how often
being contradicted flips a correct answer into a wrong one.